In [20]:
import py7zr
import re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
ZIP_DIR = Path('zips')
RAW_DIR = Path('data/raw')

def unzip_all(zip_dir, raw_dir):
    for zip_path in sorted(zip_dir.glob('*.7z')):
        year = re.search(r'20\d{2}', zip_path.name).group()
        out_dir = raw_dir / year
        out_dir.mkdir(parents=True, exist_ok=True)
        with py7zr.SevenZipFile(zip_path, 'r') as z:
            z.extractall(out_dir)
        print(f'{zip_path.name}  ->  {out_dir}')

unzip_all(ZIP_DIR, RAW_DIR)

spx_eod_2023q1-cfph7w.7z  ->  data\raw\2023
spx_eod_2023q2-kdxt36.7z  ->  data\raw\2023
spx_eod_2023q3-w9b0jk.7z  ->  data\raw\2023
spx_eod_2023q4-ai4uc9.7z  ->  data\raw\2023


In [16]:
txt_files = sorted(RAW_DIR.rglob('*.txt'))
print('Number of .txt files:', len(txt_files))
for f in txt_files:
    print(f, '|', round(f.stat().st_size / 1e6, 1), 'MB')

Number of .txt files: 12
data\raw\2023\spx_eod_202301.txt | 51.9 MB
data\raw\2023\spx_eod_202302.txt | 53.2 MB
data\raw\2023\spx_eod_202303.txt | 64.1 MB
data\raw\2023\spx_eod_202304.txt | 52.0 MB
data\raw\2023\spx_eod_202305.txt | 58.5 MB
data\raw\2023\spx_eod_202306.txt | 59.8 MB
data\raw\2023\spx_eod_202307.txt | 58.9 MB
data\raw\2023\spx_eod_202308.txt | 67.7 MB
data\raw\2023\spx_eod_202309.txt | 60.7 MB
data\raw\2023\spx_eod_202310.txt | 67.3 MB
data\raw\2023\spx_eod_202311.txt | 65.5 MB
data\raw\2023\spx_eod_202312.txt | 64.0 MB


In [ ]:
with open(txt_files[0]) as f:
    print(f.readline())
    print(f.readline())

[QUOTE_UNIXTIME], [QUOTE_READTIME], [QUOTE_DATE], [QUOTE_TIME_HOURS], [UNDERLYING_LAST], [EXPIRE_DATE], [EXPIRE_UNIX], [DTE], [C_DELTA], [C_GAMMA], [C_VEGA], [C_THETA], [C_RHO], [C_IV], [C_VOLUME], [C_LAST], [C_SIZE], [C_BID], [C_ASK], [STRIKE], [P_BID], [P_ASK], [P_SIZE], [P_LAST], [P_DELTA], [P_GAMMA], [P_VEGA], [P_THETA], [P_RHO], [P_IV], [P_VOLUME], [STRIKE_DISTANCE], [STRIKE_DISTANCE_PCT]

1672866000, 2023-01-04 16:00, 2023-01-04, 16.000000, 3853.390000, 2023-01-04, 1672866000, 0.000000, 1.000000, 0.000000, 0.000000, 0.000000, 0.000000, , 1.000000, 3041.370000, 7 x 7, 2846.800000, 2848.300000, 1000.000000, 0.000000, 0.050000, 0 x 272, 0.030000, 0.000000, 0.000000, 0.000230, -0.024810, -0.000330, 8.175360, 7.000000, 2853.400000, 0.740000



In [18]:
def read_header(path):
    with open(path) as f:
        return [c.strip().strip('[]').lower() for c in f.readline().strip().split(',')]

ref = read_header(txt_files[0])
print(len(ref), 'columns:', ref)

for f in txt_files:
    if read_header(f) != ref:
        print('DIFFERENT:', f.name)
print('Check done')

33 columns: ['quote_unixtime', 'quote_readtime', 'quote_date', 'quote_time_hours', 'underlying_last', 'expire_date', 'expire_unix', 'dte', 'c_delta', 'c_gamma', 'c_vega', 'c_theta', 'c_rho', 'c_iv', 'c_volume', 'c_last', 'c_size', 'c_bid', 'c_ask', 'strike', 'p_bid', 'p_ask', 'p_size', 'p_last', 'p_delta', 'p_gamma', 'p_vega', 'p_theta', 'p_rho', 'p_iv', 'p_volume', 'strike_distance', 'strike_distance_pct']
Check done


In [19]:
KEEP = ['quote_date', 'underlying_last', 'expire_date', 'dte', 'strike',
        'c_bid', 'c_ask', 'c_last', 'c_volume', 'c_size',
        'c_iv', 'c_delta', 'c_gamma', 'c_vega', 'c_theta', 'c_rho']

def load_calls(path):
    return pd.read_csv(path,
                       names=ref, header=0, 
                       usecols=KEEP,
                       skipinitialspace=True,     
                       low_memory=False)

calls = pd.concat([load_calls(f) for f in txt_files], ignore_index=True)
calls['quote_date'] = pd.to_datetime(calls['quote_date'])
calls['expire_date'] = pd.to_datetime(calls['expire_date'])

print(calls.shape)
print(calls.dtypes)
calls.head()

(2009083, 16)
quote_date         datetime64[us]
underlying_last           float64
expire_date        datetime64[us]
dte                       float64
c_delta                   float64
c_gamma                   float64
c_vega                    float64
c_theta                   float64
c_rho                     float64
c_iv                      float64
c_volume                  float64
c_last                    float64
c_size                        str
c_bid                     float64
c_ask                     float64
strike                    float64
dtype: object


,quote_date,underlying_last,expire_date,dte,c_delta,c_gamma,c_vega,c_theta,c_rho,c_iv,c_volume,c_last,c_size,c_bid,c_ask,strike
0,2023-01-04,3853.39,2023-01-04,0.0,1.0,0.0,0.0,0.0,0.0,NaN,1.0,3041.37,7 x 7,2846.8,2848.3,1000.0
1,2023-01-04,3853.39,2023-01-04,0.0,1.0,0.0,0.0,0.0,0.0,NaN,1.0,0.00,7 x 7,2646.8,2648.3,1200.0
2,2023-01-04,3853.39,2023-01-04,0.0,1.0,0.0,0.0,0.0,0.0,NaN,NaN,0.00,1 x 1,2427.6,2471.0,1400.0
3,2023-01-04,3853.39,2023-01-04,0.0,1.0,0.0,0.0,0.0,0.0,NaN,1.0,2417.29,7 x 7,2246.8,2248.3,1600.0
4,2023-01-04,3853.39,2023-01-04,0.0,1.0,0.0,0.0,0.0,0.0,NaN,NaN,0.00,7 x 7,2046.8,2048.3,1800.0


In [21]:

calls['mid'] = (calls['c_bid'] + calls['c_ask']) / 2

calls['moneyness'] = calls['underlying_last'] / calls['strike']

print('First date:', calls['quote_date'].min().date())
print('Last date: ', calls['quote_date'].max().date())
print('Trading days:', calls['quote_date'].nunique())

calls[['mid', 'underlying_last', 'strike', 'moneyness', 'dte']].describe()

First date: 2023-01-04
Last date:  2023-12-29
Trading days: 249


,mid,underlying_last,strike,moneyness,dte
count,2.008906e+06,2.009083e+06,2.009083e+06,2.009083e+06,2.009083e+06
mean,5.639718e+02,4.296016e+03,3.962464e+03,1.259653e+00,1.260212e+02
std,7.529731e+02,2.295852e+02,1.008334e+03,1.285603e+00,2.283716e+02
min,2.500000e-02,3.808190e+03,1.000000e+02,3.173492e-01,0.000000e+00
25%,5.090000e+01,4.118310e+03,3.590000e+03,9.756085e-01,1.600000e+01
50%,2.677000e+02,4.317960e+03,4.075000e+03,1.046306e+00,4.400000e+01
75%,7.608000e+02,4.478550e+03,4.450000e+03,1.189069e+00,1.449600e+02
max,4.617900e+03,4.783830e+03,1.200000e+04,4.719460e+01,2.199000e+03
